In [1]:
import sys 
from pathlib import Path 
import pandas as pd 

ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
paths = get_paths(ROOT)

paths

ProjectPaths(root=WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526'), src=WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/src'), notebooks=WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/notebooks'), data_processed=WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/data/processed'), data_raw=WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/data/raw'), checkpoints=WindowsPath('C:/Users/cola0/Desktop/nlp.project-colangelo-2526/checkpoints'))

In [2]:
train_path = paths.data_processed / "train.parquet"
val_path = paths.data_processed / "val.parquet"
test_path = paths.data_processed / "test.parquet"

train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)
test_df = pd.read_parquet(test_path)

print(len(train_df), len(val_df), len(test_df))
print("pos rate:", train_df["label"].mean(), val_df["label"].mean(), test_df["label"].mean())
train_df.head()

467362 52251 54300
pos rate: 0.2629717435307107 0.2629806128112381 0.26298342541436465


,review_id,movie_id,text,label
0,56845,tt0012349,"""The Kid"" is a powerfully emotional and wonder...",1
1,56846,tt0012349,The Kid became a critically hailed internation...,1
2,56847,tt0012349,A tramp finds an abandoned kid on the street. ...,1
3,56848,tt0012349,The Kid is a comedy film about a baby abandone...,1
4,56849,tt0012349,It was one of the first few movies of 'The Tra...,1


In [3]:
from windowing import build_tokenizer, create_chunks_dataframe 

TOKENIZER_NAME = "bert-base-uncased"
MAX_LEN = 256
STRIDE = 128 

tok = build_tokenizer(TOKENIZER_NAME)

subset = train_df.sample(n=min(200, len(train_df)), random_state=42).copy()
chunks_test = create_chunks_dataframe(subset, tok, max_len=MAX_LEN, stride=STRIDE)

chunks_test.head(), len(chunks_test)

c:\Users\cola0\anaconda3\envs\nlp-project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(   review_id   movie_id  label  chunk_index  \
 0        491  tt0111161      1            0   
 1       2233  tt0111161      0            0   
 2       4025  tt0111161      0            0   
 3       4166  tt0111161      0            0   
 4       4982  tt0068646      0            0   
 
                                            input_ids  \
 0  [101, 1045, 2031, 6404, 2129, 2116, 2335, 1045...   
 1  [101, 2023, 3185, 2003, 1996, 3284, 6758, 2144...   
 2  [101, 6140, 3931, 2009, 2003, 1037, 4326, 7615...   
 3  [101, 2028, 2305, 2043, 1045, 2001, 11347, 207...   
 4  [101, 2023, 2143, 2428, 2441, 2026, 2159, 2000...   
 
                                       attention_mask  \
 0  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...   
 1  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...   
 2  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...   
 3  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...   
 4  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...   
 
                      

In [4]:
g = chunks_test.groupby("review_id")["chunk_index"].max() +1

print(f"chunks per review: mean = {g.mean()}, median = {g.median()}, max = {g.max()}")

chunk_lens = chunks_test["attention_mask"].apply(sum)
print(f"chunk token len: mean = {chunk_lens.mean()} , median = {chunk_lens.median()} , max = {chunk_lens.max()}")

assert (g >= 1).all()

chunks per review: mean = 2.09, median = 1.0, max = 10
chunk token len: mean = 218.9019138755981 , median = 256.0 , max = 256


In [ ]:
from windowing import WindowConfig, build_and_save_split_chunks

cfg = WindowConfig(
    tokenizer_name=TOKENIZER_NAME, 
    max_len=MAX_LEN, 
    stride=STRIDE
    )

out_train = paths.data_processed / "train_chunks.parquet"
out_val   = paths.data_processed / "val_chunks.parquet"
out_test  = paths.data_processed / "test_chunks.parquet"

build_and_save_split_chunks(train_path, out_train, cfg)
build_and_save_split_chunks(val_path, out_val, cfg)
build_and_save_split_chunks(test_path, out_test, cfg)

list(paths.data_processed.iterdir())
